In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd
from scipy import stats

TASK_DIR = Path.cwd().parent
INPUT_DIR = TASK_DIR / "input"
OUTPUT_DIR = TASK_DIR / "output"

FIT_PATH = INPUT_DIR / "arw4_fit.json"
TRAJECTORIES_PATH = OUTPUT_DIR / "trajectories.npy"
SUMMARY_PATH = OUTPUT_DIR / "simulation_summary.csv"
MANIFEST_PATH = OUTPUT_DIR / "simulation_manifest.json"

MODEL_NAME = "ARW4"
MODEL_TAG = "arw4"
N = 10_000
Y = 20
SEED = 63

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
fit = json.loads(FIT_PATH.read_text())
params = sorted(fit["params"],key=lambda x: x["cutoff_start"])
alpha_q0 = float(fit["alpha_q0"])

print(f"Initial exponential scale: {alpha_q0:.6f}")
display(pd.DataFrame(params)[["cutoff_start","cutoff_end","alpha","mode_beta","n"]])

In [ ]:
def sample_trunc_laplace_resample_list(loc, scale, trunc):
    result = stats.laplace(scale=scale,loc=loc).rvs()
    to_resample = np.where(result < trunc)[0]
    while len(to_resample) > 0:
        resampled = stats.laplace(scale=scale,loc=loc[to_resample]).rvs()
        result[to_resample] = resampled
        to_resample = np.where(result < trunc)[0]
    return result

def first_above_k(arr, k, biggest=20):
    for a in arr:
        if k < a: return a
    return biggest

def simulate_trajectories(params, alpha_q0, n=10_000, Y=20):
    trajectories = []
    q_last = stats.expon.rvs(scale=alpha_q0,size=n)
    trajectories.append(q_last)
    career_stage = 0
    career_stages = sorted([x["cutoff_start"] for x in params])[1:] + [np.inf]
    for year in range(Y):
        next_cutoff = first_above_k(career_stages,year)
        if next_cutoff > career_stages[career_stage]: career_stage += 1
        current_params = params[career_stage]
        mode = q_last * current_params["mode_beta"]
        q_next = sample_trunc_laplace_resample_list(mode,current_params["alpha"],0)
        trajectories.append(q_next); q_last = q_next
    return np.array(trajectories)

In [ ]:
np.random.seed(SEED)
trajs = simulate_trajectories(params,alpha_q0,N,Y)
np.save(TRAJECTORIES_PATH,trajs)

summary = pd.DataFrame({"CareerAge": np.arange(Y + 1),"mean_pubs_adj": trajs.mean(axis=1),"median_pubs_adj": np.median(trajs,axis=1),"sd_pubs_adj": trajs.std(axis=1,ddof=1),"zero_fraction": (trajs == 0).mean(axis=1),"minimum_pubs_adj": trajs.min(axis=1),"maximum_pubs_adj": trajs.max(axis=1),"mean_log_pubs_adj": np.log(trajs + .49).mean(axis=1),"var_log_pubs_adj": np.log(trajs + .49).var(axis=1,ddof=1)})
summary.to_csv(SUMMARY_PATH,index=False)
MANIFEST_PATH.write_text(json.dumps({"model": MODEL_NAME,"model_tag": MODEL_TAG,"created_utc": datetime.now(timezone.utc).isoformat(),"seed": SEED,"n": N,"Y": Y,"fit": str(FIT_PATH),"trajectories": str(TRAJECTORIES_PATH)},indent=2))

print(f"Trajectories: {trajs.shape}")
display(summary.loc[summary.CareerAge.isin([0,5,10,20])])